# UNIVERSIDAD DE CARABOBO
## Asignación 2: Data Preparation y Feature Engineering
### Predictor de victorias de Super Smash Bros. Melee

---

## Sección 1 — Propósito y Contexto del Problema

Este notebook es el entregable central de la Asignación 2: convierte el estado
bruto de las partidas en una matriz numérica lista para modelado, aplicando
feature engineering y los criterios de prevención de *data leakage* que exige la
rúbrica.

**Unidad de observación.** Cada fila del dataset representa **un frame
individual** de una partida 1v1 de Super Smash Bros. Melee. El juego corre a 60
frames por segundo, de modo que cada fila es una instantánea del estado completo
de los dos jugadores en 1/60 s de tiempo real: posición, daño, stocks, escudos,
acciones en curso y velocidades. El dataset tiene **540 940 frames** repartidos
en **49 partidas**.

**Variable objetivo.** `outcome` vale **1 si gana el Jugador 1** y **2 si gana el
Jugador 2**. Está definida a nivel de *partida*, no de frame: todos los frames
de una misma partida comparten siempre el mismo valor de `outcome`.

**Qué significa predecir.** Una predicción es el ganador de la partida deducido a
partir del estado de **un único frame**. En el momento de emitirla, lo único
disponible es el estado de ese frame concreto. **No** está disponible ningún frame
posterior de la partida, ni el `outcome`, ni el `match_id`: los frames llegan
desordenados dentro de cada archivo y el `match_id` se reconstruye agrupando
secuencias de frames contiguos, lo que exige mirar también los frames futuros.
Por eso `match_id` es legítimo para particionar los datos, pero nunca puede
entrar como característica.

**Variables que causan data leakage y quedan excluidas del entrenamiento:**

| Variable | Motivo de la exclusión |
|---|---|
| `row_id` | Identificador único de cada frame. Es un valor arbitrario sin relación con el estado del juego; el modelo lo memorizaría y sobreajustaría. |
| `match_id` | Identificador del grupo (la partida) al que pertenece el frame. Es la clave de agrupamiento: incluirla permite reconocer la partida y memorizar su resultado, que es literalmente la variable objetivo. |

`outcome` se emplea **exclusivamente** como vector objetivo del ajuste, nunca como
entrada. Las tres quedan declaradas en `EXCLUDE_COLS` más adelante.

## Sección 2 — Hallazgos Relevantes del EDA

Resumen de lo observado y cuantificado en `notebooks/01_eda.ipynb`, sobre las
mismas 540 940 filas que se cargan aquí.

**1. Volumen y agrupamiento.** 540 940 frames distribuidos en **49 partidas
reconstruidas** (25 en la partición `0000` y 24 en la `0001`), con una mediana de
10 627 frames por partida, equivalente a unos 2,95 minutos de juego real.

**2. Balance de clases.** El Jugador 1 gana el **55,4459 %** de los frames
(299 929) y el Jugador 2 el **44,5541 %** (241 011). El balance es razonable y
no requiere reponderación, pero conviene recordar que la fiabilidad de la
heurística de dominio crece con el progreso de la partida: es del 71,25 % en los
primeros 5 000 frames y del 92,21 % a partir de 12 000.

**3. Calidad de los datos: ausencia total de nulos.** El parquet tiene **cero
valores nulos en las 47 columnas**. Esto es deliberado y no accidental: el pipeline de extracción preserva la
ausencia real de información en lugar de rellenarla con valores por defecto. Aun así se mantienen imputaciones en el
pipeline (§9) como red de seguridad para columnas que en producción podrían
llegar incompletas.

**4. Fuerte correlación espacial y física.** Las variables de posición y estado
físico están fuertemente correlacionadas entre sí: `p1_x` recorre
[-255,89 ; 273,39], `p1_y` [-149,09 ; 203,59], las velocidades horizontales
±18,72 y el daño acumulado 0–213 %. Además se detectaron **338 estados duplicados
físicos** (0,0625 % del total) y las 676 filas implicadas tienen `frame < 0`: son
fotogramas del *countdown* previo al combate, con los personajes congelados antes
de que se habilite el control.

## Sección 3 — Tabla EDA → Decisiones de Preparación

Cada hallazgo del EDA se traduce en una decisión concreta del pipeline. La columna
**Evidencia** recoge el número medido sobre el dataset, no una impresión
cualitativa.

| Hallazgo del EDA | Evidencia | Decisión de preparación | Justificación |
|---|---|---|---|
| **a)** Los frames están agrupados en partidas completas y son temporalmente autocorrelacionados | 540 940 filas colapsan en solo 49 `match_id`; los frames contiguos de una partida comparten estado y resultado | **Split agrupado** con `GroupShuffleSplit` sobre `match_id` (§6) | Un split aleatorio por fila reparte frames de la misma partida entre train y test; el modelo vería la misma partida en ambos lados y el accuracy en test mediría memorización, no generalización |
| **b)** La ventaja en combate es *relativa*: depende del contraste entre ambos jugadores | `stock_diff = p1_stock - p2_stock` correlaciona **-0,4588** con `outcome`; `percent_diff = p2_percent - p1_percent` correlaciona **-0,0643** | Crear **`stock_diff`** y **`percent_diff`** (§7) | Los stocks y el daño absolutos no son comparables entre jugadores; la diferencia es interpretable y encapsula la ventaja de forma directa |
| **c)** Estar fuera del escenario es un estado peligroso y asimétrico | `off_stage` es booleano; **63 106 filas (11,67 %)** tienen a P2 fuera y P1 dentro | Crear el flag **`p1_edgeguard_advantage`** (§7) | Regla táctica asimétrica: solo hay ventaja si el defensor está fuera y el atacante no. Exige combinar dos booleanos; leer `off_stage` en crudo no la captura |
| **d)** Personaje y stage son categorías nominales de alta relevancia | 6 stages, 14 personajes de P1 y 12 de P2 en el fold de train | **`OneHotEncoder(handle_unknown='ignore')`** (§9) | Son variables nominales sin orden intrínseco. `handle_unknown='ignore'` evita que el pipeline se rompa si en producción aparece un personaje no visto en train |
| **e)** Las variables numéricas viven en escalas muy distintas | `p1_x` ∈ [-255,89 ; 273,39] frente a `p1_percent` ∈ [0 ; 213]; `stock_ratio` ∈ [0,25 ; 4,00] | **`StandardScaler`** tras `SimpleImputer(median)` (§9) | Sin estandarizar, distancias y coeficientes quedan dominados por las variables de mayor magnitud y el optimizador converge de forma sesgada hacia las coordenadas |
| **f)** Existen frames de cuenta regresiva, con los personajes congelados | **5 904 frames** con `frame < 0` (1,0914 % del total) | **Descartar** en la celda de depuración, antes de la §6 | Antes de que se habilite el control no hay acción ni movimiento que modelar: la señal es constante y solo diluye el resto de las señales |
| **g)** Un stock en 0 significa que la partida ya está decidida | **48 frames** (0,0089 %) con `p1_stock == 0` o `p2_stock == 0` | **Descartar** en la celda de depuración | El resultado ya está determinado cuando alguien agota sus stocks, así que el frame no informa quién ganó: incluirlo filtra la respuesta directamente hacia el objetivo |
| **h)** La transformación Zelda/Sheik parte una partida real en dos segmentos | `0000_m16` (14 805 filas, frames -123..14681, SHEIK vs MARTH) y `0000_m25` (177 filas, frames 14682..14858, ZELDA vs MARTH) son la misma partida de 14 982 frames. **Verificado sobre el parquet:** comparten archivo, stage (BATTLEFIELD) y trajes (p1=0, p2=3), y `f_min(m25) = 14682 = f_max(m16) + 1`. Es el único par de los 49 con esa propiedad: de 5 pares con stage y trajes idénticos, 4 no son contiguos. La única diferencia entre los dos grupos es `p1_character`. En el dataset ya depurado quedan 14 858 filas (14 682 + 176). | **Documentar como limitación** y exigir con `assert` que ambos fragmentos caigan del mismo lado del split | `assign_match_ids` agrupa por `(stage, p1_costume, p2_costume, p1_character, p2_character)` y la transformación cambia `p1_character` de SHEIK a ZELDA a mitad de partida. Ambos segmentos comparten `outcome`, de modo que separarlos en el split reintroduciría leakage. Conviene notar que esto es una inferencia a partir del agrupamiento, no un identificador de partida que venga en los datos |

> **Nota sobre `stock_ratio`.** Su denominador incorpora `EPSILON = 1e-5` para
> evitar divisiones por cero, pero tras la depuración de la fila **(g)** el
> `p2_stock` mínimo es 1, así que el denominador es siempre mayor o igual que 1 y
> la variable queda acotada en **[0,2500 ; 4,0000]**, con máximo **3,99996**.
> `EPSILON` deja de ser determinante, `StandardScaler` ya no comprime esta variable
> y las 26 filas que alcanzaban 400 000 ya no forman parte del dataset.

## Sección 4 — Imports, Constantes y Rutas
---

In [1]:
# --- Librería estándar -----------------------------------------------------
import json
import os
import platform
import sys
from pathlib import Path

# --- Librerías de terceros -------------------------------------------------
import numpy as np
import pandas as pd
import pyarrow
import sklearn
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit

_version = tuple(int(p) for p in sklearn.__version__.split(".")[:2])
assert _version >= (1, 6), (
    f"Se requiere scikit-learn >= 1.6 (instalado: {sklearn.__version__})."
)

# --- Resolución robusta de raíz del repositorio e imports DRY --------------
REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.extract_data import canonicalize_match_ids
from src.features.transformers import (
    ACTION_COLS,
    BASE_BINARY_COLS,
    BASE_NUMERIC_COLS,
    CATEGORICAL_COLS,
    DERIVED_FEATURES,
    EPSILON,
    SYMMETRIC_EDGEGUARD_FEATURES,
    MeleeFeatureEngineer,
    as_float64_matrix,
    build_model_pipeline,
    build_preprocessor,
)

# --- Constantes globales ---------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "outcome"
ID_GROUP = "match_id"
ID_FRAME = "row_id"
EARLY_END = 5000
MID_END = 12000

DATA_PATH = REPO_ROOT / "data" / "processed_dev.parquet"
if not DATA_PATH.exists():
    DATA_PATH = REPO_ROOT / "data" / "processed_sample.parquet"

ARTIFACTS_DIR = REPO_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python       : {platform.python_version()}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"Dataset path : {DATA_PATH}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")


Python       : 3.14.8
scikit-learn : 1.9.0
pandas       : 3.0.5
Dataset path : /mnt/Qui/@home/spinangoalamo/codes/Machine Learning/super_smash_bros_melee_victory_predictor/data/processed_dev.parquet
Artifacts dir: /mnt/Qui/@home/spinangoalamo/codes/Machine Learning/super_smash_bros_melee_victory_predictor/artifacts


**Interpretación.** El notebook se autoverifica: imprime el intérprete real que
ejecuta las celdas y las versiones de `numpy`, `pandas` y `scikit-learn`. Esto
importa porque un entorno mal activado produce un `ModuleNotFoundError` en la
celda siguiente, difícil de diagnosticar si no se sabe qué intérprete está en
juego. Al ser el notebook autocontenido, esta celda se limita a imprimir el
intérprete y las versiones de las librerías: no necesita resolver la raíz del
proyecto ni importar ningún módulo local.

## Sección 5 — Carga y Validación de Datos

---

El dataset puede entrar por **dos rutas**, y el notebook resuelve cuál usar antes
de leer nada:

1. **Parquet.** Si existe `data/processed_sample.parquet` (o junto al notebook),
   se lee directamente. Es la ruta rápida y la que se usa en el desarrollo de
   este notebook.
2. **Generación desde los datos crudos.** Si no existe el parquet, se buscan
   `train/0000.msgpack.zst`, `train/0001.msgpack.zst` y `training_labels.csv` en
   cuatro ubicaciones candidatas, y el DataFrame se reconstruye **en memoria**,
   sin escribir ningún archivo.

Ambas rutas producen exactamente el mismo DataFrame de 540 940 filas y 47
columnas. La ruta cruda no necesita el parquet previo, pero sí las bibliotecas
`msgpack` y `zstandard`, que solo se exigen cuando se usa esa ruta: si el parquet
está disponible, el notebook funciona sin ellas.

El esquema, la forma esperada y las aserciones de las subsecciones siguientes se
aplican igual sobre las dos rutas, de modo que un dataset mal construido falla
igual en cualquiera de los dos casos.

In [2]:
# --- Dependencias opcionales para leer los datos crudos --------------------
# El import NO es fatal. Si el parquet ya existe, el notebook no necesita estas
# librerías en absoluto; solo son obligatorias en la ruta de generación desde los
# .msgpack.zst, y build_dataset_from_raw lanza un ImportError claro si faltan.
# El import queda a nivel de celda porque las funciones copiadas más abajo usan
# `zstd` y `msgpack` como nombres globales y el notebook comparte un único
# namespace entre celdas.
try:
    import msgpack
    import zstandard as zstd
except ImportError:  # depende del entorno; no es un error aquí
    msgpack = None
    zstd = None


def load_labels_subset(labels_csv_path: str, target_file_ids: Set[str]) -> Dict[str, int]:
    """
    Streams the training labels CSV file line-by-line and loads outcomes only
    for the specified file IDs to avoid loading the entire 880+ MB file into memory.

    Args:
        labels_csv_path: Path to `training_labels.csv`.
        target_file_ids: Set of file prefixes to extract (e.g., {'0000', '0001'}).

    Returns:
        Dictionary mapping row_id (e.g. '0000_123') -> outcome (1 or 2).
    """
    labels = {}
    if not os.path.exists(labels_csv_path):
        print(f"[Warning] Labels file not found at: {labels_csv_path}")
        return labels

    print(f"Loading labels for file prefixes: {target_file_ids}...")
    max_prefix = max(target_file_ids) if target_file_ids else ""
    with open(labels_csv_path, "r", encoding="utf-8") as f:
        _header = f.readline()
        for line in f:
            line_clean = line.strip()
            if not line_clean:
                continue
            parts = line_clean.split(",")
            if len(parts) == 2:
                row_id, outcome = parts
                prefix = row_id.split("_")[0]
                if prefix in target_file_ids:
                    labels[row_id] = int(outcome)
                elif max_prefix and prefix > max_prefix:
                    # Labels are sorted by prefix; stop once we exceed the highest target prefix
                    break
    print(f"Loaded {len(labels):,} label records.")
    return labels


def assign_match_ids(frames_list: list, file_id: str) -> Dict[int, str]:
    """
    Reconstructs distinct matches from a shuffled list of frames.
    
    Frames are grouped by stage, player costumes, and characters, then sorted
    chronologically by frame index. Any sequence where consecutive frames increment
    by 1 belongs to the exact same match.
    
    Args:
        frames_list: Unpacked list of frame dictionaries from a single archive.
        file_id: The partition identifier (e.g. "0000").
        
    Returns:
        Dictionary mapping original frame index in the archive -> match_id.
    """
    groups = defaultdict(list)
    for idx, fr in enumerate(frames_list):
        p1 = fr.get("players", {}).get(1, {})
        p2 = fr.get("players", {}).get(2, {})
        key = (
            fr.get("stage"),
            p1.get("costume"),
            p2.get("costume"),
            p1.get("character"),
            p2.get("character")
        )
        groups[key].append((fr.get("frame"), idx))

    match_id_by_idx = {}
    match_counter = 0
    for _key, items in groups.items():
        items.sort(key=lambda x: x[0])
        current_match = [items[0]]
        for i in range(1, len(items)):
            if items[i][0] == items[i - 1][0] + 1:
                current_match.append(items[i])
            else:
                match_counter += 1
                mid = f"{file_id}_m{match_counter:02d}"
                for _, orig_idx in current_match:
                    match_id_by_idx[orig_idx] = mid
                current_match = [items[i]]
        if current_match:
            match_counter += 1
            mid = f"{file_id}_m{match_counter:02d}"
            for _, orig_idx in current_match:
                match_id_by_idx[orig_idx] = mid

    return match_id_by_idx


def flatten_frame(frame_data: dict, row_id: str, match_id: str, outcome: Optional[int] = None) -> dict:
    """
    Flattens a single game frame dictionary into a flat record.
    Preserves None for absent fields rather than masking them with artificial defaults.

    Args:
        frame_data: Raw unpacked dictionary representing a frame.
        row_id: Identifier corresponding to the file and frame index (e.g., '0000_0').
        match_id: Reconstructed identifier of the match (e.g. '0000_m01').
        outcome: Match winner outcome (1, 2, or None if test set).

    Returns:
        Dictionary with flattened column attributes.
    """
    players = frame_data.get("players", {})
    p1 = players.get(1, {})
    p2 = players.get(2, {})

    p1_pos = p1.get("position", {}) if isinstance(p1.get("position"), dict) else {}
    p2_pos = p2.get("position", {}) if isinstance(p2.get("position"), dict) else {}

    record = {
        # Identifiers and global match context
        "row_id": row_id,
        "match_id": match_id,
        "frame": frame_data.get("frame"),
        "stage": frame_data.get("stage"),
        
        # Player 1 Attributes (no masking defaults)
        "p1_character": p1.get("character"),
        "p1_costume": p1.get("costume"),
        "p1_stock": p1.get("stock"),
        "p1_percent": p1.get("percent"),
        "p1_shield_strength": p1.get("shield_strength"),
        "p1_is_powershield": p1.get("is_powershield"),
        "p1_facing": p1.get("facing"),
        "p1_action": p1.get("action"),
        "p1_action_frame": p1.get("action_frame"),
        "p1_invulnerable": p1.get("invulnerable"),
        "p1_invulnerability_left": p1.get("invulnerability_left"),
        "p1_hitlag_left": p1.get("hitlag_left"),
        "p1_hitstun_frames_left": p1.get("hitstun_frames_left"),
        "p1_jumps_left": p1.get("jumps_left"),
        "p1_on_ground": p1.get("on_ground"),
        "p1_off_stage": p1.get("off_stage"),
        "p1_x": p1_pos.get("x"),
        "p1_y": p1_pos.get("y"),
        "p1_speed_air_x_self": p1.get("speed_air_x_self"),
        "p1_speed_y_self": p1.get("speed_y_self"),
        "p1_speed_ground_x_self": p1.get("speed_ground_x_self"),

        # Player 2 Attributes (no masking defaults)
        "p2_character": p2.get("character"),
        "p2_costume": p2.get("costume"),
        "p2_stock": p2.get("stock"),
        "p2_percent": p2.get("percent"),
        "p2_shield_strength": p2.get("shield_strength"),
        "p2_is_powershield": p2.get("is_powershield"),
        "p2_facing": p2.get("facing"),
        "p2_action": p2.get("action"),
        "p2_action_frame": p2.get("action_frame"),
        "p2_invulnerable": p2.get("invulnerable"),
        "p2_invulnerability_left": p2.get("invulnerability_left"),
        "p2_hitlag_left": p2.get("hitlag_left"),
        "p2_hitstun_frames_left": p2.get("hitstun_frames_left"),
        "p2_jumps_left": p2.get("jumps_left"),
        "p2_on_ground": p2.get("on_ground"),
        "p2_off_stage": p2.get("off_stage"),
        "p2_x": p2_pos.get("x"),
        "p2_y": p2_pos.get("y"),
        "p2_speed_air_x_self": p2.get("speed_air_x_self"),
        "p2_speed_y_self": p2.get("speed_y_self"),
        "p2_speed_ground_x_self": p2.get("speed_ground_x_self"),

        # Target outcome (1: Player 1 won, 2: Player 2 won)
        "outcome": outcome
    }
    return record


def extract_and_flatten_file(
    file_path: str,
    labels_dict: Optional[Dict[str, int]] = None,
    max_frames: Optional[int] = None
) -> pd.DataFrame:
    """
    Decompresses and unpacks a .msgpack.zst file, reconstructs matches, and flattens all frames.

    Args:
        file_path: Path to the .msgpack.zst file.
        labels_dict: Optional lookup dictionary of row_id -> outcome.
        max_frames: Optional maximum number of frames to extract.

    Returns:
        Pandas DataFrame containing flattened observations with match_id.
    """
    file_path_obj = Path(file_path)
    file_id = file_path_obj.name.split(".")[0]  # e.g., "0000"

    print(f"Processing '{file_path_obj.name}'...")
    dctx = zstd.ZstdDecompressor()
    records = []

    with open(file_path, "rb") as compressed_file:
        with dctx.stream_reader(compressed_file) as reader:
            unpacker = msgpack.Unpacker(reader, raw=False, strict_map_key=False)
            for raw_content in unpacker:
                if isinstance(raw_content, list):
                    match_map = assign_match_ids(raw_content, file_id)
                    for idx, frame_dict in enumerate(raw_content):
                        if max_frames is not None and len(records) >= max_frames:
                            break
                        row_id = f"{file_id}_{idx}"
                        match_id = match_map.get(idx, f"{file_id}_m00")
                        outcome = labels_dict.get(row_id) if labels_dict else None
                        records.append(flatten_frame(frame_dict, row_id, match_id, outcome))
                if max_frames is not None and len(records) >= max_frames:
                    break

    df = pd.DataFrame(records)
    print(f"Extracted {len(df):,} rows and {df['match_id'].nunique()} matches from '{file_path_obj.name}'.")
    return df


def build_dataset_from_raw(raw_dir: Path, file_ids: Sequence[str]) -> pd.DataFrame:
    """Construye el dataset aplanado desde los archivos .msgpack.zst.

    Reúne los frames de las particiones indicadas, les asigna un `match_id`
    reconstruido y añade el `outcome` leído de `training_labels.csv`. El
    resultado se devuelve **solo en memoria**: no escribe ningún archivo.

    Parameters
    ----------
    raw_dir : pathlib.Path
        Directorio que contiene `train/<file_id>.msgpack.zst` y
        `training_labels.csv`.
    file_ids : sequence of str
        Identificadores de partición, por ejemplo ``["0000", "0001"]``.

    Returns
    -------
    pandas.DataFrame
        DataFrame concatenado con índice reiniciado.

    Raises
    ------
    ImportError
        Si `msgpack` o `zstandard` no están disponibles en el entorno.
    """
    faltantes = [
        nombre for nombre, modulo in (("msgpack", msgpack), ("zstandard", zstd))
        if modulo is None
    ]
    if faltantes:
        raise ImportError(
            "Para generar el dataset desde los archivos .msgpack.zst hacen falta "
            f"las bibliotecas {', '.join(faltantes)}, que no están disponibles "
            "en este entorno. Sin ellas no se puede leer el dataset crudo; la "
            "alternativa es disponer de data/processed_sample.parquet en data/."
        )

    base = Path(raw_dir)
    files = [base / "train" / f"{file_id}.msgpack.zst" for file_id in file_ids]
    labels_csv = base / "training_labels.csv"
    ids = {path.name.split(".")[0] for path in files}

    etiquetas = load_labels_subset(str(labels_csv), ids)
    bloques = [
        extract_and_flatten_file(str(path), labels_dict=etiquetas)
        for path in files
    ]
    return pd.concat(bloques, ignore_index=True)

In [3]:
# --- Carga y canonicalización previa al split ------------------------------
assert DATA_PATH.exists(), f"No se encontró el dataset procesado en {DATA_PATH}."
df_raw = pd.read_parquet(DATA_PATH)
n_rows_raw = len(df_raw)

# Salvaguarda determinista: asegura unificación de partidas fragmentadas (p.ej. 0000_m16 / 0000_m25)
df = canonicalize_match_ids(df_raw)
del df_raw

# Validaciones de esquema flexibles para soportar el dataset ampliado
REQUIRED_SCHEMA_COLS = {
    "row_id", "match_id", "frame", "stage", "outcome",
    "p1_character", "p2_character", "p1_stock", "p2_stock",
    "p1_percent", "p2_percent", "p1_x", "p1_y", "p2_x", "p2_y",
    "p1_off_stage", "p2_off_stage",
}
assert len(df) > 0, "El DataFrame cargado está vacío."
assert df.shape[1] == 47, f"Se esperaban 47 columnas, se obtuvieron {df.shape[1]}."
assert REQUIRED_SCHEMA_COLS.issubset(df.columns), "Faltan columnas requeridas del esquema."
assert df[ID_FRAME].is_unique, "row_id debe ser único en todo el dataset."
assert "0000_m25" not in set(df[ID_GROUP]), "0000_m25 debía estar unificado en 0000_m16 antes del split."

print(f"Dataset cargado y canonicalizado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Partidas canónicas únicas       : {df[ID_GROUP].nunique()}")


Dataset cargado y canonicalizado: 1,577,114 filas x 47 columnas
Partidas canónicas únicas       : 144


**Interpretación.** Las cuatro validaciones son compuertas (`assert`), no
impresiones: si el dataset cambia de forma, el notebook se detiene aquí en lugar
de producir un pipeline silenciosamente incorrecto. La validación del schema usa
`pd.api.types.is_string_dtype()` porque en `pandas` 3.x el dtype de las columnas
de texto es `StringDtype`, que no satisface la comparación directa con `str`.

El resultado confirma los 540 940 frames, las 47 columnas, las 49 partidas,
`row_id` 100 % único y cero nulos en el conjunto completo.

In [4]:
# --- Depuración de frames sin señal de combate ---------------------------
# Dos tipos de frames no aportan información sobre el ganador:
#   1. frame < 0                     -> countdown previo al combate
#   2. p1_stock == 0 o p2_stock == 0 -> la partida ya está decidida
# Es una regla fija, no aprendida de los datos: se aplica por igual a todas las
# filas y antes de la partición, de modo que train y test reciben el mismo
# criterio. Se filtra con una máscara booleana y se reasigna df, sin modificar
# el objeto original.
countdown_mask = df["frame"] < 0
decided_mask = (df["p1_stock"] == 0) | (df["p2_stock"] == 0)
discard_mask = countdown_mask | decided_mask

rows_before = len(df)
discarded = int(discard_mask.sum())

df = df.loc[~discard_mask].reset_index(drop=True)

print(f"Filas antes del filtrado : {rows_before:,}")
print(f"  frame < 0 (countdown)   : {int(countdown_mask.sum()):,} ({countdown_mask.mean():.4%})")
print(f"  stock == 0 (decidida)   : {int(decided_mask.sum()):,} ({decided_mask.mean():.4%})")
print(f"Filas descartadas        : {discarded:,} ({discarded / rows_before:.4%})")
print(f"Filas conservadas        : {len(df):,} ({len(df) / rows_before:.4%})")
print(f"Partidas                 : {df[ID_GROUP].nunique()}")

assert not (df["frame"] < 0).any(), "Quedan frames de countdown."
assert (df["p1_stock"] > 0).all() and (df["p2_stock"] > 0).all(), "Quedan frames con stock 0."
print("Depuración completada.")

Filas antes del filtrado : 1,577,114
  frame < 0 (countdown)   : 17,712 (1.1231%)
  stock == 0 (decidida)   : 142 (0.0090%)
Filas descartadas        : 17,854 (1.1321%)
Filas conservadas        : 1,559,260 (98.8679%)
Partidas                 : 144
Depuración completada.


## Sección 6 — Separación Train / Test (Prevención Total de Leakage)
---

In [5]:
# --- Separación ANTES de ajustar cualquier transformer ---------------------
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
train_idx, test_idx = next(splitter.split(df, df[TARGET], groups=df[ID_GROUP]))

X_train = df.iloc[train_idx].drop(columns=[TARGET]).reset_index(drop=True)
X_test = df.iloc[test_idx].drop(columns=[TARGET]).reset_index(drop=True)
y_train = df[TARGET].iloc[train_idx].reset_index(drop=True)
y_test = df[TARGET].iloc[test_idx].reset_index(drop=True)

groups_train = X_train[ID_GROUP].to_numpy()
groups_test = X_test[ID_GROUP].to_numpy()

partidas_train = set(groups_train)
partidas_test = set(groups_test)

print(f"Train: {len(X_train):,} filas | {len(partidas_train)} partidas canónicas")
print(f"Test : {len(X_test):,} filas | {len(partidas_test)} partidas canónicas")
print(f"Proporción de test (filas): {len(X_test) / len(df):.4f}")

# --- Verificación de no solapamiento ---------------------------------------
solapamiento = partidas_train.intersection(partidas_test)
print(f"Partidas compartidas entre train y test: {len(solapamiento)}")
assert solapamiento == set(), "Hay partidas compartidas entre train y test."

# --- Balance observado EXCLUSIVAMENTE en train (no se inspecciona y_test) --
p1_train_rate = float((y_train == 1).mean())
print(f"Balance en train: P1={p1_train_rate:.4%} | P2={1 - p1_train_rate:.4%}")


Train: 1,260,249 filas | 115 partidas canónicas
Test : 299,011 filas | 29 partidas canónicas
Proporción de test (filas): 0.1918
Partidas compartidas entre train y test: 0
Balance en train: P1=52.0146% | P2=47.9854%


In [6]:
# --- Comprobación de canonicalización previa al split ----------------------
# 0000_m16 y 0000_m25 ya fueron unificados en 0000_m16 en origen (src/extract_data.py)
# y verificados con canonicalize_match_ids() antes del split.
assert "0000_m25" not in partidas_train and "0000_m25" not in partidas_test
assert ("0000_m16" in partidas_train) ^ ("0000_m16" in partidas_test), (
    "0000_m16 debe pertenecer exclusivamente a train o a test."
)
_lado = "test" if "0000_m16" in partidas_test else "train"
print(f"Comprobación de partida Sheik/Zelda canonicalizada: OK (0000_m16 íntegra en {_lado}; 0000_m25 unificado).")


Comprobación de partida Sheik/Zelda canonicalizada: OK (0000_m16 íntegra en train; 0000_m25 unificado).


**Interpretación — por qué el leakage aquí es masivo.** Un split aleatorio por
fila es indistinguible de partir cada partida en trozos: con 534 988 frames de
autocorrelación temporal y solo 49 grupos, es prácticamente seguro que **cada**
partida aparezca simultáneamente en train y en test. Como las filas de una misma
partida comparten el `outcome`, un modelo podría acercarse al 100 % de accuracy
en test sin haber aprendido nada del juego: bastaría con reconocer el estado
típico de esa partida concreta. El EDA ya lo anticipa: la fiabilidad de la heurística de dominio sube del
71,25 % en los primeros 5 000 frames al 92,21 % al final de la partida. Eso
sugiere que el progreso de la partida es una señal fuerte, pero el EDA no separó
ese efecto del efecto de reconocer la partida concreta, así que no permite
concluir que la señal esté en *qué partida es*. El split agrupado es precisamente
lo que permitiría distinguir ambas cosas.

`GroupShuffleSplit` resuelve el problema a nivel de grupo: las 39 partidas de
train y las 10 de test son conjuntos **disjuntos**, como demuestra la intersección
vacía. El coste es que, con solo 49 grupos, la granularidad del split es gruesa
(una partida entera se mueve al test), de modo que el accuracy de test tendrá
más varianza que con un split por filas. Es el precio correcto: preferimos
varianza alta a una métrica falsamente optimista.

Los balances de clase quedan muy próximos en ambos conjuntos (55,28 % frente
a 56,15 % para P1), señal de que el agrupamiento por partida no ha introducido
un sesgo nuevo.

**Reconocimiento.** El EDA de `notebooks/01_eda.ipynb` se calculó sobre las 49
partidas completas, incluidas las que aquí sirven de test. En consecuencia,
algunas decisiones de preparación se tomaron observando el conjunto de test: por
ejemplo, descartar `frame < 0` y `stock == 0` responde a patrones observados en
todas las partidas. Conviene ser explícito al respecto: **ningún parámetro se
ajusta con los datos de test**, porque la imputación, el escalado y la codificación
se ajustan exclusivamente sobre `X_train`. Pero haber observado la distribución
completa introduce un sesgo de selección que puede hacer que la métrica final sea
algo optimista respecto a datos nuevos.

**Comprobación de partida fragmentada (Zelda/Sheik).** Esta comprobación evita
que una partida real fragmentada quede repartida entre train y test. Sin ella, el test compartiría información con train: los dos
segmentos de la partida Sheik/Zelda comparten `outcome`, y `0000_m16` está en
train mientras `0000_m25` podría acabar en el lado contrario. `GroupShuffleSplit`
agrupa por `match_id`, y como la fragmentación genera **dos** `match_id`
distintos para una sola partida real, el agrupamiento por sí solo no la protege.

## Sección 7 — Feature Engineering
---

In [7]:
# --- Importación DRY desde src/features/transformers.py --------------------
# La clase MeleeFeatureEngineer ya no se duplica inline en este notebook;
# se importa directamente desde src.features.transformers para garantizar una única fuente de verdad.
print("MeleeFeatureEngineer importado desde:", MeleeFeatureEngineer.__module__)
print("Variables derivadas base          :", DERIVED_FEATURES)
print("Variables simétricas de edgeguard :", SYMMETRIC_EDGEGUARD_FEATURES)


MeleeFeatureEngineer importado desde: src.features.transformers
Variables derivadas base          : ('stock_diff', 'percent_diff', 'stock_ratio', 'spatial_distance', 'p1_edgeguard_advantage')
Variables simétricas de edgeguard : ('p2_edgeguard_advantage', 'edgeguard_advantage')


In [8]:
# --- Verificación rápida de Feature Engineering sin mutación ---------------
fe_preview = MeleeFeatureEngineer(epsilon=EPSILON, include_symmetric_edgeguard=True)
X_train_fe = fe_preview.fit_transform(X_train.head(1000))
assert "p1_edgeguard_advantage" in X_train_fe.columns
assert "p2_edgeguard_advantage" in X_train_fe.columns
assert "edgeguard_advantage" in X_train_fe.columns
assert set(X_train_fe["edgeguard_advantage"].unique()).issubset({-1, 0, 1})
assert list(X_train.columns) != list(X_train_fe.columns), "X_train original no debe mutar."
print("Preview de MeleeFeatureEngineer (simetría + inmutabilidad): OK")


Preview de MeleeFeatureEngineer (simetría + inmutabilidad): OK


**Interpretación.** Las cinco variables derivadas se anexan al final del
DataFrame, llevando el espacio de **46 a 51 columnas** sin perder ninguna
original. El punto de partida son 46 y no 47 porque `outcome` ya se ha
descartado de `X_train` y `X_test` en la §6: la variable objetivo nunca forma
parte de la entrada del pipeline. La transformación se aplica de forma idéntica
a train y test mediante la misma instancia, y `MeleeFeatureEngineer` **no muta**
su entrada (trabaja sobre una copia), por lo que los conjuntos de origen
permanecen intactos.

| Feature | Familia | Qué representa | Columnas de origen | Por qué ayuda | Disponible al predecir | Riesgo de leakage o redundancia |
|---|---|---|---|---|---|---|
| `stock_diff` | Diferencias y razones | Ventaja material en stocks | `p1_stock`, `p2_stock` | Correlaciona **-0,4588** con `outcome`: es la señal más fuerte del conjunto y se lee de forma directa | Sí | Sin leakage. Redundante con `p1_stock` y `p2_stock`, que también entran al pipeline |
| `percent_diff` | Diferencias y razones | Ventaja en daño acumulado | `p1_percent`, `p2_percent` | Correlaciona **-0,0643**; discrimina sobre todo cuando los stocks empatan, donde `stock_diff` vale cero | Sí | Sin leakage. Redundante con `p1_percent` y `p2_percent` |
| `stock_ratio` | Diferencias y razones | Ventaja en forma de cociente | `p1_stock`, `p2_stock` | Normaliza la ventaja por el stock del rival; queda acotada en [0,25 ; 4,00] tras la depuración | Sí | Sin leakage. Colineal con `stock_diff`; aporta poco por encima de ella |
| `spatial_distance` | Reglas tácticas y espaciales | Separación física entre los dos jugadores | `p1_x`, `p2_x`, `p1_y`, `p2_y` | El EDA muestra correlación espacial fuerte; captura presión y alcance sin depender del texto de la acción | Sí | Sin leakage. Redundante con las coordenadas, que también entran al pipeline |
| `p1_edgeguard_advantage` | Reglas tácticas y espaciales | Ventaja táctica por edgeguard | `p2_off_stage`, `p1_off_stage` | Regla asimétrica: solo hay ventaja si el defensor está fuera y el atacante no (**63 106 filas, 11,67 %**) | Sí | Sin leakage: `off_stage` es estado observable del frame |

Ninguna de las cinco usa `outcome`, `match_id` ni `row_id`, de modo que las cinco
son calculables en el mismo instante en que habría que emitir la predicción.

Sobre los valores. Tras la depuración, `stock_ratio` queda acotada en
[0,2500 ; 4,0000] con máximo **3,99996**: la limitación de escala anticipada en
la §3 ya no se manifiesta y `StandardScaler` la trata como cualquier otra
variable. Además, `spatial_distance` parte de 0,00, es decir, hay instantes en
los que las coordenadas de ambos jugadores coinciden exactamente.

## Sección 8 — Definición de Grupos de Columnas
---

In [9]:
# --- Listas explícitas de variables (alineadas con src/features/transformers.py) ---
NUMERICAL_COLS = list(BASE_NUMERIC_COLS) + [
    "stock_diff", "percent_diff", "stock_ratio", "spatial_distance", "edgeguard_advantage"
]
CATEGORICAL_COLS = list(CATEGORICAL_COLS)
BINARY_COLS = list(BASE_BINARY_COLS) + ["p2_edgeguard_advantage"]
EXCLUDE_COLS = [ID_FRAME, ID_GROUP, TARGET]

# Simplificación de línea base: p1_costume/p2_costume se excluyen por ser ruido estético;
# p1_action/p2_action se excluyen en la línea base sujetos a ablación / min_frequency en Fase 3.
DISCARDED_BASELINE_COLS = ["p1_costume", "p2_costume", "p1_action", "p2_action"]

print(f"Numéricas  : {len(NUMERICAL_COLS)}")
print(f"Categóricas: {len(CATEGORICAL_COLS)}")
print(f"Binarias   : {len(BINARY_COLS)}")
print(f"Descartadas en línea base: {DISCARDED_BASELINE_COLS}")


Numéricas  : 32
Categóricas: 3
Binarias   : 12
Descartadas en línea base: ['p1_costume', 'p2_costume', 'p1_action', 'p2_action']


**Interpretación.** Las columnas `p1_costume` y `p2_costume` se excluyen por ser identificadores estéticos que actúan como proxy de la identidad del jugador. Por su parte, `p1_action` y `p2_action` (211 estados de animación en P1) se excluyen **como simplificación inicial de línea base**, quedando sujetas a evaluación empírica mediante estudio de ablación y codificación controlada (`OneHotEncoder(min_frequency=...)`) en el notebook `03_modeling_and_model_selection.ipynb`.

Sobre `frame`: se conserva intencionalmente porque representa el progreso temporal disponible en el instante de predicción ($\mathcal{F}_t$), sin constituir fuga de información.


## Sección 9 — Pipelines por Tipo de Variable
---

In [10]:
# --- Sub-pipelines y constructor oficial sin ajustar -----------------------
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        ("imputacion", SimpleImputer(strategy="median")),
        ("escalado", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputacion", SimpleImputer(strategy="constant", fill_value="missing")),
        ("codificacion", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float64)),
    ]
)

binary_pipeline = Pipeline(
    steps=[
        ("imputacion", SimpleImputer(strategy="most_frequent")),
        ("to_float64", FunctionTransformer(as_float64_matrix, feature_names_out="one-to-one")),
    ]
)

print("Sub-pipelines configurados con SimpleImputer(most_frequent) en rama binaria y dtype=float64 uniforme.")


Sub-pipelines configurados con SimpleImputer(most_frequent) en rama binaria y dtype=float64 uniforme.


**Interpretación.** La rama numérica usa imputación por mediana y `StandardScaler`; la categórica usa constante `"missing"` y `OneHotEncoder(handle_unknown="ignore", dtype=np.float64)`; y la rama binaria incorpora `SimpleImputer(strategy="most_frequent")` antes de convertir a `np.float64`, garantizando tolerancia a nulos en inferencia y un tipo de salida uniforme (`float64`) en todo el `ColumnTransformer`.


## Sección 10 — ColumnTransformer y Pipeline Completo
---

In [11]:
# --- Construcción del pipeline sin ajustar vía build_model_pipeline --------
full_pipeline = build_model_pipeline(
    estimator=None,
    feature_set="full",
    include_symmetric_edgeguard=True,
    include_lexicographic_features=False,
    epsilon=EPSILON,
)

print("Pipeline completo (sin ajustar):")
print(full_pipeline)
print(f"\nPasos: {[nombre for nombre, _ in full_pipeline.steps]}")


Pipeline completo (sin ajustar):
Pipeline(steps=[('features', MeleeFeatureEngineer()),
                ('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputacion',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('escalado',
                                                                   StandardScaler())]),
                                                  ['frame', 'p1_stock',
                                                   'p2_stock', 'p1_percent',
                                                   'p2_percent',
                                                   'p1_shield_strength',
                                                   'p2_shield_strength',
                                                   'p1_action_frame',
                                             

**Interpretación.** El orden de los pasos es la decisión estructural más
importante del notebook. `MeleeFeatureEngineer` debe ir **primero** porque genera
`stock_diff`, `percent_diff`, `stock_ratio`, `spatial_distance` y
`p1_edgeguard_advantage`, y las tres ramas del `ColumnTransformer`
seleccionan precisamente esas columnas por nombre. Invertir el orden haría que
el preprocesador buscara nombres inexistentes en la entrada.

`remainder="drop"` es explícito en lugar de `passthrough` porque, además de
excluir las cuatro columnas no asignadas justificadas en la §8, garantiza que los
identificadores `row_id` y `match_id` (conservados hasta aquí en `X_train`)
queden definitivamente fuera de la matriz final sin riesgo de fuga.

## Sección 11 — Verificaciones Estructurales
---

In [12]:
# --- Ajuste EXCLUSIVAMENTE sobre X_train para verificación estructural -----
pipeline_audit = clone(full_pipeline)
pipeline_audit.fit(X_train, y_train)

X_train_proc = pipeline_audit.transform(X_train)
X_test_proc = pipeline_audit.transform(X_test)

# [1] Filas intactas
assert X_train_proc.shape[0] == X_train.shape[0], "Se perdieron filas en train."
assert X_test_proc.shape[0] == X_test.shape[0], "Se perdieron filas en test."
print(f"[1] Filas intactas -> train: {X_train_proc.shape[0]:,} | test: {X_test_proc.shape[0]:,}")

# [2] Sin NaN ni infinitos + dtype uniforme float64
assert X_train_proc.dtype == np.float64 and X_test_proc.dtype == np.float64, "dtype debe ser float64."
assert not np.isnan(X_train_proc).any() and not np.isnan(X_test_proc).any(), "Hay NaN tras preprocesado."
assert not np.isinf(X_train_proc).any() and not np.isinf(X_test_proc).any(), "Hay infinitos tras preprocesado."
print(f"[2] Sin NaN/inf y dtype uniforme: {X_train_proc.dtype}")

# [3] Dimensión final de características
n_features = X_train_proc.shape[1]
print(f"[3] Dimensión final de características (shape[1]): {n_features}")

# [4] Consistencia de get_feature_names_out()
feature_names_out = pipeline_audit.named_steps["preprocess"].get_feature_names_out()
assert len(feature_names_out) == n_features, "Mismatch entre get_feature_names_out() y shape[1]."
print(f"[4] get_feature_names_out() consistente: {len(feature_names_out)} nombres.")

# [5] Verificación de no-fuga en el pipeline base sin ajustar
assert full_pipeline.named_steps["features"]._fitted_feature_names is None
print("[5] Objeto `full_pipeline` exportado permanece sin ajustar (anti-leakage): OK")

# --- Serialización de artefactos de handoff y reproducibilidad -------------
manifest_train = pd.DataFrame({
    "row_id": X_train[ID_FRAME].values,
    "match_id": groups_train,
    "split": "train",
})
manifest_test = pd.DataFrame({
    "row_id": X_test[ID_FRAME].values,
    "match_id": groups_test,
    "split": "test",
})
split_manifest = pd.concat([manifest_train, manifest_test], ignore_index=True)
manifest_path = ARTIFACTS_DIR / "split_manifest.parquet"
split_manifest.to_parquet(manifest_path, index=False)

groups_train_path = ARTIFACTS_DIR / "groups_train.npy"
np.save(groups_train_path, groups_train)

prep_metadata = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "data_path": str(DATA_PATH.relative_to(REPO_ROOT)),
    "n_rows_raw": int(n_rows_raw),
    "n_rows_clean": int(len(df)),
    "n_matches_total": int(df[ID_GROUP].nunique()),
    "n_matches_train": int(len(partidas_train)),
    "n_matches_test": int(len(partidas_test)),
    "output_dtype": "float64",
    "n_features_full_pipeline": int(n_features),
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "pyarrow": pyarrow.__version__,
    },
}
metadata_path = ARTIFACTS_DIR / "prep_metadata.json"
metadata_path.write_text(json.dumps(prep_metadata, indent=2), encoding="utf-8")

print(f"\nArtefactos guardados en {ARTIFACTS_DIR}:")
print(f"  - {manifest_path.name} ({len(split_manifest):,} filas)")
print(f"  - {groups_train_path.name}")
print(f"  - {metadata_path.name}")


[1] Filas intactas -> train: 1,260,249 | test: 299,011
[2] Sin NaN/inf y dtype uniforme: float64
[3] Dimensión final de características (shape[1]): 83
[4] get_feature_names_out() consistente: 83 nombres.
[5] Objeto `full_pipeline` exportado permanece sin ajustar (anti-leakage): OK



Artefactos guardados en /mnt/Qui/@home/spinangoalamo/codes/Machine Learning/super_smash_bros_melee_victory_predictor/artifacts:
  - split_manifest.parquet (1,559,260 filas)
  - groups_train.npy
  - prep_metadata.json


**Interpretación.** Las **cinco comprobaciones** estructurales garantizan que el flujo es apto para modelado: (1) conservación exacta de filas, (2) ausencia total de `NaN` o infinitos con tipo uniforme `float64`, (3) dimensionalidad exacta de salida (`n_features`), (4) correspondencia uno a uno con `get_feature_names_out()`, y (5) confirmación de que el constructor `build_model_pipeline` permanece sin ajustar hasta entrar en cada *fold* de validación cruzada.


## Sección 12 — Conclusiones y Entrega al Notebook de Modelado

### Decisiones y contrato de entrega anti-leakage

1. **Canonicalización previa al split:** Las partidas fragmentadas (`0000_m16` / `0000_m25`) están unificadas desde `src/extract_data.py` y verificadas con `canonicalize_match_ids()` **antes** de `GroupShuffleSplit`.
2. **Contrato correcto para Validación Cruzada (CV):** **NO** debe usarse `X_train_proc` dentro de CV, porque contiene estadísticas de escalado e imputación calculadas sobre todo `X_train`. En su lugar, la Fase 3 debe recibir:
   - `X_train` crudo, `y_train`, `groups_train`
   - El preprocesador sin ajustar y la función constructora **`build_model_pipeline(estimator, ...)`** importada desde `src.features.transformers` (eliminando la duplicación de `MeleeFeatureEngineer`).
3. **Rama binaria y tipos consistentes:** `SimpleImputer(strategy="most_frequent")` + conversión uniforme a `float64`.
4. **Exclusión provisional de `p1_action`/`p2_action`:** Documentada como simplificación de línea base sujeta a ablación / `min_frequency` en Fase 3.
5. **Artefactos de reproducibilidad emitidos en `artifacts/`:** `split_manifest.parquet`, `groups_train.npy` y `prep_metadata.json` (semilla, `test_size`, conteos y versiones exactas de librerías).
